In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import gdown
import re
import os
import pandas as pd
import seaborn as sns
import scvi
import gc
import scipy as sp
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)
from cell2location.utils.filtering import filter_genes

def colour_tables(feature_series):
    features = list(set(feature_series))
    hex_features = []
    for i in range(len(features)):
        r = np.random.randint(0,200)
        g = np.random.randint(0,200)
        b = np.random.randint(0,200)
        colour = '#{:02x}{:02x}{:02x}'.format(r, g, b)
        hex_features.append(colour)
    return hex_features

# Load and prepare adata

In [ ]:
# Load data
adata = sc.read_h5ad('../data/adata/GBM_LEAP_15_05_23_coarse_annos.h5ad')
adata.X = adata.layers['counts'].copy()

# CNA analysis subsetting

## Load CNAs

In [ ]:
CNA_info = []
CNA_path = '../data/RNA_CNA/'
for root, dirs, files in os.walk(CNA_path, topdown=False):
    for f in files:
        if 'AT' in f:
                CNA_info.append(pd.read_csv(root+f))
CNA_info = pd.concat(CNA_info, ignore_index=True)

In [ ]:
adata.obs = adata.obs.reset_index().merge(CNA_info[['cell_id', 'cnv_leiden', 'cnv_score', 
                                                    'CNV_signal_sqsum', 'CNV_signal_mean', 
                                                    'cnv_corr']], 
                                          on = 'cell_id', how = 'left').set_index('index')

In [ ]:
adata.obs['corr_threshold'] = np.where(adata.obs['cnv_corr']>0.3,1,0)
adata.obs['signal_threshold'] = np.where(adata.obs['CNV_signal_mean']>0.02, 1, 0)
adata.obs['malignant_signal'] = np.where((adata.obs['corr_threshold']==1) & 
                                         (adata.obs['signal_threshold']), 1, 0)

## Evaluate filters

In [ ]:
sc.pl.umap(adata, color=['cnv_score', 'CNV_signal_mean', 'malignant_signal'], 
               gene_symbols='SYMBOL',
               legend_fontsize=14, frameon=False, ncols=1, vmax = [0.01, 0.01,1])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
summary = adata.obs.groupby(['leiden_scVI', 'malignant_signal'])[['cell_id']]\
    .count().reset_index()\
    .pivot(index='leiden_scVI', columns='malignant_signal', values='cell_id')
summary['perc_CNA'] = 100*(summary[1]/(summary[0]+summary[1]))
summary = summary.reset_index()

plt.axhspan(0,3, color = 'lightgreen')
plt.axhspan(3,100, color = 'lightcoral')

sns.scatterplot(data=summary, x = 'leiden_scVI', y = 'perc_CNA')
for i, leiden in enumerate (summary['leiden_scVI']):
    plt.annotate(leiden, (int(summary.loc[i]['leiden_scVI'])+0.7, summary.loc[i]['perc_CNA']+0.5) )

plt.show()

In [ ]:
adata.obs['malignant_cluster'] = np.where(adata.obs['leiden_scVI']\
                            .isin(summary[summary['perc_CNA']>3]['leiden_scVI']), 1, 0)

In [ ]:
# generate UMAP
sc.pl.umap(
    adata,
    color=['malignant_cluster', 'coarse_prediction'],
    ncols=2,
    size=0.1,
    frameon=False)

# Iterative leiden clustering for malignant assignment

In [ ]:
def prep_for_integration(ad):
    ad = ad.raw.to_adata()
    ad.raw = ad.copy()
    ad.layers['counts'] = ad.X.copy()

    # i.e. scanpy/seurat methods
    sc.pp.normalize_total(ad, target_sum=1e4)
    sc.pp.log1p(ad)

    sc.pp.highly_variable_genes(ad, 
                                min_mean=0.0075, 
                                max_mean=4, 
                                min_disp=0.1,
                                batch_key="donor_id"
    )
    selected = ad.var['highly_variable']
    ad = ad[:, selected].copy()
    return ad


def scVI_reintegrate(ad):
    
    ad = ad.copy()
    
    # check the UMAP without phase as a covariate
    scvi.model.SCVI.setup_anndata(
        ad,
        layer="counts",
        batch_key = "sample",
        categorical_covariate_keys=["date", "donor_id", "site_id"]
    )

    model = scvi.model.SCVI(ad,
                           n_hidden = 1024,
                           n_layers = 2,
                           n_latent = 50,
                           gene_likelihood = 'nb',
                           dispersion = 'gene-batch',
                           use_observed_lib_size = True)
    model.train(train_size=0.99, batch_size = 1024) #max_epochs=200

    latent = model.get_latent_representation()
    ad.obsm["X_scVI"] = latent

    # use scVI latent space for UMAP generation
    sc.pp.neighbors(ad, use_rep="X_scVI")
    sc.tl.leiden(ad, key_added="leiden_scVI_test", resolution = 2)
    return ad

In [ ]:
## Prep integration 
adata_tme = prep_for_integration(adata[adata.obs['malignant_cluster']==0,:])
adata_gbm = prep_for_integration(adata[adata.obs['malignant_cluster']==1,:])

## Get ambiguous clusters
import seaborn as sns
import matplotlib.pyplot as plt

cluster = 'leiden_scVI_test'
CNA_sorted = {}
non_malignant = []
malignant = []
i = 0

while ((len(non_malignant)>0) | (len(malignant)>0)) | (i==0):
    i+=1
    iteration = "iteration_{}".format(i)
    print(iteration)
    
    # MALIGNANT
    if (len(non_malignant)>0) | (i==1):
        print('Running malignant')
        adata_gbm = scVI_reintegrate(adata_gbm)

        summary = adata_gbm.obs.groupby([cluster, 'malignant_signal'])[['cell_id']]\
            .count().reset_index()\
            .pivot(index=cluster, columns='malignant_signal', values='cell_id')
        summary['perc_CNA'] = 100*(summary[1]/(summary[0]+summary[1]))
        summary = summary.reset_index()

        non_malignant = summary[summary['perc_CNA']<5][cluster].tolist()
        for cl in non_malignant:
            adata_gbm.obs['check_{}'.format(cl)] = np.where(adata_gbm.obs[cluster]==cl, 'Check', '')

            # generate UMAP
            sc.pl.umap(
                adata_gbm,
                color=['check_{}'.format(cl)],
                ncols=1,
                size=0.1,
                frameon=False)
        gbm_addition = adata_gbm[adata_gbm.obs[cluster].isin(non_malignant)]
        CNA_sorted["{}_gbm".format(iteration)] = gbm_addition

        # Filter iteration
        adata_gbm = adata_gbm[~adata_gbm.obs[cluster].isin(non_malignant)]
    else:
        print('Malignant cells reached threshold')
    
    # TME
    if (len(malignant)>0) | (i==1):
        print('Running TME')
        adata_tme = scVI_reintegrate(adata_tme)

        summary = adata_tme.obs.groupby([cluster, 'malignant_signal'])[['cell_id']]\
            .count().reset_index()\
            .pivot(index=cluster, columns='malignant_signal', values='cell_id')
        summary['perc_CNA'] = 100*(summary[1]/(summary[0]+summary[1]))
        summary = summary.reset_index()

        malignant = summary[summary['perc_CNA']>=3][cluster].tolist()
        for cl in malignant:
            adata_tme.obs['check_{}'.format(cl)] = np.where(adata_tme.obs[cluster]==cl, 'Check', '')

            # generate UMAP
            sc.pl.umap(
                adata_tme,
                color=['check_{}'.format(cl)],
                ncols=1,
                size=0.1,
                frameon=False)
        tme_addition = adata_tme[adata_tme.obs[cluster].isin(malignant)]
        CNA_sorted["{}_tme".format(iteration)] = tme_addition
    else:
        print('TME cells reached threshold')

## Export

In [ ]:
ambiguous = sc.concat(CNA_sorted.values(), merge = 'same')

In [ ]:
adata[adata.obs['cell_id'].isin(adata_tme.obs['cell_id'])]\
    .write_h5ad('../data/adata/GBM_LEAP_TME_only.h5ad')
adata[adata.obs['cell_id'].isin(adata_gbm.obs['cell_id'])]\
    .write_h5ad('../data/adata/GBM_LEAP_malignant_only.h5ad')
adata[adata.obs['cell_id'].isin(ambiguous.obs['cell_id'])]\
    .write_h5ad('../data/adata/GBM_LEAP_ambiguous_only.h5ad')